In [1]:
import mne
import numpy as np
import matplotlib.pyplot as plt

In [2]:
from pathlib import Path
PROJECT_DIR = Path.cwd().parent  # project root (assumes notebook run from notebooks/)

In [3]:
raw=mne.io.read_raw_fif(f"{PROJECT_DIR}/Data/Raw/sample_audvis_raw.fif",preload=True)
print(raw.info)

Opening raw data file /home/gerko/eeg_blender_project/Data/Raw/sample_audvis_raw.fif...
    Read a total of 3 projection items:
        PCA-v1 (1 x 102)  idle
        PCA-v2 (1 x 102)  idle
        PCA-v3 (1 x 102)  idle
    Range : 25800 ... 192599 =     42.956 ...   320.670 secs
Ready.
Reading 0 ... 166799  =      0.000 ...   277.714 secs...
<Info | 21 non-empty values
 acq_pars: ACQch001 110113 ACQch002 110112 ACQch003 110111 ACQch004 110122 ...
 bads: 2 items (MEG 2443, EEG 053)
 ch_names: MEG 0113, MEG 0112, MEG 0111, MEG 0122, MEG 0123, MEG 0121, MEG ...
 chs: 204 Gradiometers, 102 Magnetometers, 9 Stimulus, 60 EEG, 1 EOG
 custom_ref_applied: False
 description: acquisition (megacq) VectorView system at NMR-MGH
 dev_head_t: MEG device -> head transform
 dig: 146 items (3 Cardinal, 4 HPI, 61 EEG, 78 Extra)
 events: 1 item (list)
 experimenter: MEG
 file_id: 4 items (dict)
 highpass: 0.1 Hz
 hpi_meas: 1 item (list)
 hpi_results: 1 item (list)
 lowpass: 172.2 Hz
 meas_date: 2002-12-

In [4]:
raw_eeg=raw.pick('eeg')
print("EEG channels:",len(raw_eeg.ch_names))
print ("channel name:",raw_eeg.ch_names)


EEG channels: 60
channel name: ['EEG 001', 'EEG 002', 'EEG 003', 'EEG 004', 'EEG 005', 'EEG 006', 'EEG 007', 'EEG 008', 'EEG 009', 'EEG 010', 'EEG 011', 'EEG 012', 'EEG 013', 'EEG 014', 'EEG 015', 'EEG 016', 'EEG 017', 'EEG 018', 'EEG 019', 'EEG 020', 'EEG 021', 'EEG 022', 'EEG 023', 'EEG 024', 'EEG 025', 'EEG 026', 'EEG 027', 'EEG 028', 'EEG 029', 'EEG 030', 'EEG 031', 'EEG 032', 'EEG 033', 'EEG 034', 'EEG 035', 'EEG 036', 'EEG 037', 'EEG 038', 'EEG 039', 'EEG 040', 'EEG 041', 'EEG 042', 'EEG 043', 'EEG 044', 'EEG 045', 'EEG 046', 'EEG 047', 'EEG 048', 'EEG 049', 'EEG 050', 'EEG 051', 'EEG 052', 'EEG 053', 'EEG 054', 'EEG 055', 'EEG 056', 'EEG 057', 'EEG 058', 'EEG 059', 'EEG 060']


In [5]:
data, times = raw_eeg.get_data(return_times=True)
print("Data shape:", data.shape)
print("Duration:", times[-1], "seconds")

Data shape: (60, 166800)
Duration: 277.7136813300495 seconds


In [6]:
bands = {
    'delta': (0.5, 4),
    'theta': (4, 8),
    'alpha': (8, 13),
    'beta':  (13, 30),
    'gamma': (30, 100)
}


In [7]:
from scipy.signal import stft

sfreq = raw_eeg.info['sfreq']
window_size = int(sfreq * 2) # 2 sec 

freqs_stft, _, Zxx = stft(data, fs=sfreq, nperseg=window_size)

print("STFT shape:", Zxx.shape)
print("Frequencies:", len(freqs_stft))


STFT shape: (60, 601, 279)
Frequencies: 601


In [8]:
band_power_time = {}
for band_name, (fmin, fmax) in bands.items():
    freq_mask = (freqs_stft >= fmin) & (freqs_stft <= fmax)
    band_power_time[band_name] = np.abs(Zxx[:, freq_mask, :]).mean(axis=1)
    print(f"{band_name}: {band_power_time[band_name].shape}")

delta: (60, 279)
theta: (60, 279)
alpha: (60, 279)
beta: (60, 279)
gamma: (60, 279)


In [9]:
for band_name, power in band_power_time.items():
    power_norm = (power - power.min()) / (power.max() - power.min())
    np.save(f"{PROJECT_DIR}/Data/processed/EEG/{band_name}_power.npy", power_norm.astype(np.float32))
    print(f"Saved {band_name}_power.npy, range: {power_norm.min():.3f} to {power_norm.max():.3f}")

Saved delta_power.npy, range: 0.000 to 1.000
Saved theta_power.npy, range: 0.000 to 1.000
Saved alpha_power.npy, range: 0.000 to 1.000
Saved beta_power.npy, range: 0.000 to 1.000
Saved gamma_power.npy, range: 0.000 to 1.000


In [10]:
montage = raw_eeg.get_montage()
print(montage)

<DigMontage | 78 extras (headshape), 4 HPIs, 3 fiducials, 60 channels>


In [11]:
positions = montage.get_positions()
ch_pos = positions['ch_pos']

for ch_name, pos in list(ch_pos.items())[:5]:
    print(ch_name, pos)


EEG 001 [-0.03737009  0.10568011  0.07333875]
EEG 002 [-0.01491282  0.11079782  0.07650087]
EEG 003 [0.01212589 0.10959283 0.07736051]
EEG 004 [-0.06572473  0.0781765   0.06546284]
EEG 005 [-0.0506296   0.08445242  0.08830147]


In [12]:
ch_names = list(ch_pos.keys())
electrode_pos = np.array([ch_pos[ch] for ch in ch_names], dtype=np.float32)

print("Erode positions shape:", electrode_pos.shape)
np.save(f"{PROJECT_DIR}/Data/processed/EEG/electrode_positions.npy", electrode_pos)


Electrode positions shape: (60, 3)
Saved!


In [13]:
from scipy.spatial import KDTree

lh_verts_centered = np.load(f"{PROJECT_DIR}/Data/processed/MRT/lh_pial.npy")
rh_verts_centered = np.load(f"{PROJECT_DIR}/Data/processed/MRT/rh_pial.npy")

all_verts = np.vstack([lh_verts_centered, rh_verts_centered])

print("Erode center:", electrode_pos.mean(axis=0))
print("Pial center:", all_verts.mean(axis=0))

Electrode center: [-0.00965965  0.00522629  0.09177694]
Pial center: [ 5.9696578e-07  3.9082448e-07 -7.6069739e-08]


In [14]:
# Center electrodes first, then scale to match pial coordinate space
electrode_pos_centered = electrode_pos - electrode_pos.mean(axis=0)
electrode_scale = all_verts.max() / electrode_pos_centered.max()
electrode_pos_scaled = electrode_pos_centered * electrode_scale

print("Scaled Erode range:", electrode_pos_scaled.min(), "to", electrode_pos_scaled.max())
print("Pial range:", all_verts.min(), "to", all_verts.max())


Scaled electrode range: -0.8061697 to 0.9074476
Pial range: -0.76270974 to 0.9074476


In [15]:
### AI codeing help Start  
from scipy.spatial import KDTree

lh_count = len(lh_verts_centered)
all_verts = np.vstack([lh_verts_centered, rh_verts_centered])

# Split electrodes by X position into LH, RH and midline
midline_tol = 0.10
lh_elec_mask = electrode_pos_scaled[:, 0] < -midline_tol
rh_elec_mask = electrode_pos_scaled[:, 0] > midline_tol
mid_elec_mask = ~lh_elec_mask & ~rh_elec_mask

lh_electrodes = electrode_pos_scaled[lh_elec_mask | mid_elec_mask]
rh_electrodes = electrode_pos_scaled[rh_elec_mask | mid_elec_mask]

print(f"LH electrodes: {lh_elec_mask.sum()}, RH electrodes: {rh_elec_mask.sum()}, Midline: {mid_elec_mask.sum()}")

# Distance mask per hemisphere using only its own electrodes
lh_tree = KDTree(lh_electrodes)
rh_tree = KDTree(rh_electrodes)

lh_dist, lh_nearest = lh_tree.query(lh_verts_centered, k=5)
rh_dist, rh_nearest = rh_tree.query(rh_verts_centered, k=5)

# Smooth fade zone instead of hard cutoff
fade_start = 0.25
fade_end = 0.35

lh_falloff = np.clip((fade_end - lh_dist[:, 0]) / (fade_end - fade_start), 0, 1)
rh_falloff = np.clip((fade_end - rh_dist[:, 0]) / (fade_end - fade_start), 0, 1)

# Gaussian interpolation between electrodes
sigma = 0.15
lh_weights = np.exp(-lh_dist**2 / (2 * sigma**2))
lh_weights = lh_weights / lh_weights.sum(axis=1, keepdims=True)
lh_weights *= lh_falloff[:, np.newaxis]

rh_weights = np.exp(-rh_dist**2 / (2 * sigma**2))
rh_weights = rh_weights / rh_weights.sum(axis=1, keepdims=True)
rh_weights *= rh_falloff[:, np.newaxis]

# Map hemisphere-local electrode indices to global electrode array
lh_elec_indices = np.where(lh_elec_mask | mid_elec_mask)[0]
rh_elec_indices = np.where(rh_elec_mask | mid_elec_mask)[0]

nearest_electrodes = np.vstack([lh_elec_indices[lh_nearest], rh_elec_indices[rh_nearest]])
weights = np.vstack([lh_weights, rh_weights])

np.save(f"{PROJECT_DIR}/Data/processed/EEG/vertex_nearest_electrodes.npy", nearest_electrodes.astype(np.int32))
np.save(f"{PROJECT_DIR}/Data/processed/EEG/vertex_weights.npy", weights.astype(np.float32))
### AI codeing help end

LH electrodes: 27, RH electrodes: 27, Midline: 6


In [16]:
delta_power = np.load(f"{PROJECT_DIR}/Data/processed/EEG/delta_power.npy")